# 方法详解

![](./img/1.png)

## 1. 输入数据的表示

在蛋白质中，每个残基（即已连接到多肽链上的氨基酸单元）的主链通常包含如下原子序列：$\mathrm{N} - \mathrm{C}_\alpha - \mathrm{C} - \mathrm{O}$。为了有效捕捉蛋白质结构的刚体特性和空间关系，使用frames建模这些原子坐标。具体方法为：使用一个以 $C_\alpha$ 为原点的刚体坐标系（frame），其方向由一个旋转矩阵 $R \in SO(3)$ 表征。为构造该坐标系，我们首先通过已知的三维坐标计算两个向量：$V_1 = \mathbf{C} - \mathbf{C_\alpha}$ 和 $V_2 = \mathbf{N} - \mathbf{C_\alpha}$，分别表示从 $C_\alpha$ 指向 $C$ 和 $N$ 的方向。由于这两个向量通常不正交，我们采用 Gram–Schmidt 正交化方法来生成一组正交单位向量：将 $V_1$ 归一化得到 $u_1$，再从 $V_2$ 中去除在 $u_1$ 方向上的分量并归一化，得到 $u_2$，最后通过叉乘 $u_1 \times u_2$ 得到第三个正交方向 $u_3$。最终，旋转矩阵 $r = [u_1\ u_2\ u_3]$ 是一个$3 \times 3$的正交矩阵，由三列向量组成。它表示一个旋转，将标准坐标系变换到以为$\mathrm{C}_\alpha$原点、方向由$u_1 , u_2 , u_3$构成的新坐标系。换句话说，$r$描述了残基骨架的空间朝向。



需要注意的是，我们通常不用 $C_\alpha \rightarrow O$ 的向量来构建坐标系，是因为氧原子的位置受二面角 $\psi$ 控制，不能直接从主链几何推导出。

最终模型的输入：$T=(r,x) \in SE(3)$，其中$r=GramSchmidt(v_{1},v_{2}), x=\mathrm{C}_\alpha \in \mathbb{R}^3$。

对包含$N$个残基的蛋白质序列，建模的空间为$SE(3)^N$，其中每个残基表示为一个刚体 (旋转+平移)。



## 2. 前向加噪过程

## 3. 后向去噪过程

## 4. 损失函数

## 5. 训练过程

## 6. 推理过程

## 7. 模型评估



# 1. 总览（一句话总结）

FrameDiff 把蛋白主链的每个残基表示为一个 **frame**（刚体位姿，属于 SE(3)），并在该流形上构造一个能分离“旋转（SO(3)）”与“平移（R³）”成分的前向扩散过程（rotational Brownian + translational OU/VP 风格过程），用等变网络学习时间依赖的 **score**（即 $\nabla\log p_t$），通过反向 SDE（或 Langevin 离散化）采样出新的 backbone。论文把理论（SE(3) 上的扩散/score matching）和工程（基于 AlphaFold2 的 FramePred 网络、批次策略、损失设计）结合起来，能不依赖大规模预训练直接生成设计性良好的 monomer backbone。

---

# 2. 数据建模（frames 表示、SE(3)$^N$）

核心思想与表示：

* 把长度为 $N$ 的蛋白主链用 $N$ 个 frame 表示：对第 $n$ 个残基，用旋转矩阵 $r_n\in SO(3)$（描述残基局部坐标轴方向）和平移向量 $x_n\in\mathbb{R}^3$（通常以 Cα 原子位置为基点）构成刚体 $T_n=(r_n,x_n)\in SE(3)$。整体表示为 $T=(T_1,\dots,T_N)\in SE(3)^N$。此外还预测每个残基的一个 torsion 角 $\psi_n$（用于确定 O 原子位置）。这些 frame 可以通过固定参考原子（N, Cα, C）由原子坐标构造（atom2frame）。

要点/动机：

* 用 frame 而不是直接坐标的好处：把旋转和平移分解开，便于在 Lie 群上定义自然的 Brownian（热核）以及实现 SE(3) 等变/不变建模（例如通过“中心化”实现全局 SE(3) 不变性）。

---

# 3. 前向（加噪）过程 — 在 SE(3)$^N$ 上构造的 SDE

目标：从数据分布 $p_0(T)$ 漸近变到简单参考噪声分布（旋转接近均匀、平移接近高斯），使得可以学习逆向过程。

关键设计与数学形式：

1. **选度量并把 SE(3) 视作 SO(3) × R³（从黎曼角度）**，因此可以把前向过程分解为旋转分量和平移分量的独立扩散（条件于初始值）。论文给出基于 Laplace–Beltrami 的 Brownian motion 定义，使得在 SO(3) 上有热核（heat kernel），在 R³ 上为 Ornstein–Uhlenbeck（或 VP/线性扩散）型过程。

2. **具体的连续 SDE（简化写法）**（论文给出更严格的表示；概念上）：

   * 旋转（每个残基）随时间 $t$ 按 SO(3) 上的 Brownian motion 演化；
   * 平移按带衰减的 OU / VP 风格：

     $$
     \begin{aligned}
     dR_n(t) &= \text{(SO(3) Brownian increment)} \\
     dX_n(t) &= -\tfrac{1}{2} X_n(t)\,dt + dB^{\mathbb{R}^3}_n(t)
     \end{aligned}
     $$
   * 为保证全局 SE(3) “不变性”，对平移部分在每一步做“center-of-mass free” 处理（投影矩阵 $P$ 去除质心），从而得到在子流形 $SE(3)^N_0$（中心化的 SE(3)）上的过程。最终写成（论文 Eq.(6) 的概念形式）：

     $$
     dT(t) = [0, -\tfrac12 P X(t)] dt + [dB_{SO(3)^N}(t),\, P\,dB_{\mathbb{R}^{3N}}(t)].
     $$

   这里 $P$ 表示去中心化的投影（去 CoM）。

3. **条件密度形式**：

   * 对平移：条件分布 $p_{t|0}(x(t)\mid x(0))$ 是高斯（OU/VP），均值和方差可解析表示（paper 给出与常见 VP-SDE 对应的 closed form）。
   * 对旋转：条件分布 $p_{t|0}(r(t)\mid r(0))$ = heat kernel on SO(3)，可由表示论/特征展开得到（paper 给出 Proposition 3.2 以及具体的热核展式），在实际实现中用一个 “isotropic Gaussian on SO(3)”（IGSO(3)）近似/表述。

4. **噪声 schedule（方差随 t 的变化）**：

   * 平移通常用线性 β schedule（类似 VP），旋转用一个对数/非线性 σ schedule，使得两者的方差曲线比较协同（paper 给了具体 σ\_min, σ\_max, β\_min, β\_max 的选择）。

---

# 4. 反向（去噪）过程与 score matching

目标：学习时间依赖的 score $\nabla_{T(t)}\log p_t(T(t))$，从噪声分布反演回数据分布。

1. **理论上的逆 SDE（连续形式）**（paper Eq.(7) 等式的概念）：

   * 旋转部分的逆 SDE：

     $$
     d\widehat R(t) = \nabla_{r}\log p_{T_f-t}(\widehat T(t))\,dt + dB_{SO(3)}(t),
     $$
   * 平移部分的逆 SDE（带投影 P）：

     $$
     d\widehat X(t) = P\Big(\tfrac12 \widehat X(t) + \nabla_x\log p_{T_f-t}(\widehat T(t))\Big)\,dt + P\,dB_{\mathbb R^{3N}}(t).
     $$

   其中 $\nabla_r$ 表示在 SO(3) 上的黎曼梯度（tangent space），$\nabla_x$ 是 Euclidean 梯度。

2. **训练目标 —— Denoising Score Matching (DSM)**：

   * 要逼近真实 score，网络 $s_\theta(t,\cdot)$ 被训练最小化 DSM 损失（论文 Eq.(3) 的推广）：

     $$
     \mathcal{L}(\theta) = \mathbb{E}_{t\sim U[0,T_f]}\; \lambda_t\; \big\| \nabla\log p_{t|0}(T(t)\mid T(0)) - s_\theta(t,T(t))\big\|^2,
     $$

     在 SE(3) 中，score 可分解为旋转和平移两部分，故用对应的等变网络分别估计。

3. **旋转 score 的解析形式（用以计算 DSM 的“目标”）**：

   * 对 SO(3) 有解析表达（paper Proposition 3.4），可写成与矩阵对数（matrix logarithm）有关的表达式，形式上涉及 $r(t)^\top r(0)$ 的对数映射和热核的导数（paper 给了详式）。这允许在训练样本上计算目标 score（或其近似），从而用 MSE 回归训练网络预测旋转方向中的 score。

4. **FramePred 的“预测-转换为 score”策略**：

   * 网络不是直接输出切空间的 score 向量（也可以），而是输出一个对 $T(0)$ 的去噪预测 $\widehat T(0)$（即预测无噪声的 frame），以及 torsion $\widehat\psi$；然后根据 $p_{t|0}(\cdot\mid \widehat T(0))$ 的条件密度显式计算目标 score（旋转与平移分别计算），把它作为训练目标与网络内部对应的 sθ 进行比较（paper Sec.4.1）。这和传统的“直接预测噪声”或“直接预测干净样本”的思想等价，但在 SE(3) 上更方便把旋转的 DSM 写清楚。

---

# 5. 损失函数（详细分项）

FrameDiff 使用 **多目标（multi-objective）损失**，主要包括：

1. **DSM 损失（主损失）**：

   * 分为旋转部分 $L^{(r)}_{\text{dsm}}$ 和平移部分 $L^{(x)}_{\text{dsm}}$，总和为 $L_{\text{dsm}} = L^{(r)}_{\text{dsm}} + L^{(x)}_{\text{dsm}}$。其中

     $$
     L(\theta)=\mathbb{E}_{t}\big[ \lambda^r_t\|\nabla_r\log p_{t|0}(R(t)|R(0)) - s^r_\theta(t,T(t))\|^2 + \lambda^x_t\|\nabla_x\log p_{t|0}(X(t)|X(0)) - s^x_\theta(t,T(t))\|^2 \big].
     $$
   * **权重选择**：论文沿用 Song et al. 的做法，为旋转选取 $\lambda^r_t = 1/\mathbb{E}[\|\nabla\log p_{t|0}(R(t)|R(0))\|^2]$（使得“平凡预测”误差为 1），为平移选取 $\lambda^x_t = (1-e^{-t})/e^{-t/2}$，使得平移部分的 DSM 简化为 **Cα 坐标的 MSE**：

     $$
     L^{(x)}_{\text{dsm}} = \frac{1}{N}\sum_{n=1}^N \|X_n(0) - \widehat X_n(0)\|^2.
     $$

     这在实现上非常方便且有助于稳定训练（paper Sec.4.2 详细给出）。

2. **辅助/结构损失（auxiliary losses）**：

   * 仅用 DSM 会学到合理的宏观拓扑但可能在原子级别（如键长、二面角、近邻几何）上不够精细，因此论文加入一些辅助损失（例如对原子坐标/局部几何的 MSE、torsion 角预测误差等），使生成的 backbone 在精细结构上更自然。论文在附录里给了具体实现与系数选择。

3. **与 RFdiffusion 等的比较**：

   * 其他工作（例如 RFdiffusion）有时直接对旋转矩阵做 Frobenius-norm 的复原损失 $L_F = \mathbb{E}\|R(0)-\widehat R(0)\|^2$。论文讨论并比较了这两种损失的差异，提出 DSM 在理论上与 score-based 生成相容且更有吸引力（但 LF 在某些设置也有效，论文做了消融实验）。

---

# 6. 网络架构与实现要点（FramePred）

* **总体结构**：FramePred 的 backbone 模块借鉴 AlphaFold2 的结构模块（Invariant Point Attention, IPA + Transformer 层 + BackboneUpdate）；模型以残基为节点、全连接图的边编码 pairwise 信息，做多轮迭代更新（L 层），每层输出对 frame 的更新（rotation & translation）以及边/节点 embedding 更新。最终输出 $\widehat T(0)$（去噪的 frame）和 $\widehat\psi$。这种设计带来 SO(3) 等变或不变性（IPA 保证对全局旋转/平移不敏感/等变），并能利用空间和序列两类注意力捕捉长程依赖。

* **自条件（self-conditioning）**：论文采用自条件技术（在 edge embedding 初始化中使用模型自己预测的 pairwise 距离信息），改善收敛与样本质量（与最近 diffusion/score 方法常用技巧一致）。

* **实现细节**：

  * 模型参数量（论文实验中）：约 17.4M 参数，L=4 层（论文具体 hyperparam 在 App.I.4）。
  * SDE 超参数：translation schedule = linear（$\beta_{\min}=0.1, \beta_{\max}=20$），rotation schedule = logarithmic（$\sigma_{\min}=0.1, \sigma_{\max}=1.5$）——这些值来自论文实验设置。

---

# 7. 训练流程（逐步、伪代码级别）

下面是基于论文附录（Alg.2、Alg.3）把训练流程按步骤把控要点罗列出来：

1. **准备数据**：从 PDB 筛选单体（长度范围、去除过多 loop 的条目等），构造 frame（atom2frame）并标准化（单位 nm / 或 Å 统一），得到 $T^{(0)}$ 数据集。论文使用 \~20k 条样本作为训练集。

2. **采样时间 t**：对每个训练样本随机采样时间 $t\sim U([\epsilon, T_f])$（论文用 uniform），这表示在不同时刻对同一样例做不同程度的加噪。

3. **前向噪声（单步/闭式采样）**：

   * 对每个残基独立（条件于 $T^{(0)}$）地直接从给定的条件分布采样 $R(t)\sim p_{t|0}(R(t)\mid R(0))$（SO(3)热核 / IGSO3），和 $X(t)\sim \mathcal{N}(e^{-t/2}X(0), (1-e^{-t})I)$（OU），然后对平移做中心化（投影 $P$）以移除 CoM。论文给出离散化实现细节（参见附录 Alg.2）。

4. **批次策略（TimestepBatch）**：

   * FramePred 是完全连接图结构，内存随残基数平方增长。为提高批次利用率，论文采用一种特殊 batching：一个 batch 中每个元素可以是同一条 backbone 的不同 time-step 实例（因此 batch 中每个条目长度相同、无 padding）。见论文附录的 Algorithm 2。

5. **前向网络与损失**：

   * 把 noised frames $T(t)$ 输入 FramePred（并给出 time embedding），网络输出 $\widehat T(0),\widehat\psi$（以及内部计算的 s^r\_\theta, s^x\_\theta），据此计算 DSM（旋转+平移）和辅助损失（torsion、局部坐标误差等），再对 batch 求均值。

6. **优化**：

   * 使用 Adam（论文用 lr=1e-4，$\beta_1=0.9,\beta_2=0.999$），梯度步更新 θ（论文实验训练约 1–2 周在两块 A100 GPU 上，参数与实际训练资源有关，附录给了实际训练时长与硬件）。

7. **自条件与稳定化**：

   * 采用 self-conditioning（将先前预测结果作为下一次 forward 的输入特征之一），配合损失加权 schedule（旋转、平移不同 λ\_t），有助稳定。

---

# 8. 采样（生成）流程

1. **初始采样**：从参考噪声分布采样 $T(T_f)$（旋转近似均匀/热核尾部，平移为标准高斯），并确保中心化（CoM=0）。

2. **逆向时间步离散化**：

   * 以小步长（论文给出 discretized Langevin 动力学实现）从 $t=T_f$ 向 $t=0$ 迭代，步骤大致为：

     * 在当前 $T(t)$ 上用 FramePred 预测 $\widehat T(0)$ → 计算当前的 score 估计（旋转与平移）→ 根据逆 SDE 做一次 Euler–Maruyama / Langevin 更新（并注入噪声）。
     * 每步对平移做 re-center（保持 CoM=0），避免数值漂移。
   * 当到达 $t=\epsilon$ 时做一次 final forward pass，用 $\widehat\psi$ 构建 O 等原子坐标，最终通过 atom2frame 映射得到完整主链原子坐标。详细采样伪码在附录 J.2。

3. **可选加速**：后续工作（FrameFlow）表明可以用 flow-matching 或更少采样步来加速采样（在保持质量的同时显著减少步数），这是 FrameDiff 的后续改进方向。([arXiv][1])

---

# 9. 实践要点与经验性选择

* **中心化（去 CoM）很重要**：因为没有在 R^{3N} 上存在完全集中的 SE(3) 不变概率测度，center-of-mass-free 处理能实现全局 SE(3) 不变性（理论与实践上都关键）。
* **旋转与平移的 schedule 需要配合**：paper 推荐对平移用线性 β，对旋转用对数 σ，使两者的噪声增长曲线更协调（实验表明这样样本略好）。
* **损失选择**：DSM 在理论上与 score-based 生成一致，能够更好地与反向 SDE 绑定；但在实际工程中可以和简单的矩阵差损失（LF）做消融比较。
* **计算/内存**：完全连接的 IPA + Transformer 使得内存随着 $N^2$ 增长，论文用特殊 batch 策略缓解；训练仍然需要较大的显存/算力（论文给出用两块 A100 的记录）。

---

# 10. 关键参考（原文 & 扩展）

* Jason Yim 等，*SE(3) diffusion model with application to protein backbone generation*（FrameDiff，ICML 2023）— **主论文与所有数学/实现细节**。
* 官方实现仓库（参考实现）： jasonkyuyim/se3\_diffusion（GitHub），可以看训练/采样代码与细节实现。([GitHub][2])
* 后续工作 FrameFlow（SE(3) flow matching，加速采样）— 可供对比与借鉴。([arXiv][1])
* FrameDiPT（FrameDiff 在结构修补/修复上的延伸）— 应用扩展。([生物预印本][3])


